# TorchRL Modules

In this tutorial, we will delve into the core functionality of TorchRL in terms of policy construction. We will primarily focus on how to build policies for different environments, and a little about how to setup the agent to explore and exploit.

This notebook was modified from the official tutorials provided by torchRL: 

**Get started with TorchRL's modules**
Author: `Vincent Moens <https://github.com/vmoens>`
url: https://docs.pytorch.org/rl/stable/tutorials/getting-started-1.html

### Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#!pip install tensordict
#!pip install torchrl

### TensorDictModules

Similar to how environments interact with instances of TensorDict, the modules used to represent policies and value functions also do the same. The core idea is simple: encapsulate a standard Module (or any other function) within a class that knows which entries need to be read and passed to the module, and then records the results with the assigned entries. 

To illustrate this, we will use the simplest policy possible: a deterministic map from the observation space to the action space. For maximum generality, we will use a LazyLinear module with the Pendulum environment we instantiated in the previous tutorial.



### Descrete action spaces

In [3]:
import torch
from tensordict.nn import TensorDictModule
from torchrl.envs import GymEnv
from torch import nn

env = GymEnv("Pendulum-v1")

#just the shape (torch)
env_actions = env.action_spec.shape
print(f"Action space: {env_actions}")

#just the shape (number)
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")



Action space: torch.Size([1])
Action space: 1


In [4]:
env.action_spec

BoundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True)),
    device=cpu,
    dtype=torch.float32,
    domain=continuous)

In [5]:
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

Observation space: 3


In [6]:
#create a model: takes observations as inputs, and outputs action
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

policy = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action"],
)



In [7]:
from torchrl.modules import Actor

#create a model: takes observations as inputs, and outputs action
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#simple to use the actor module (does the same as the TensorDict above)
policy = Actor(model)


 This is all that’s required to execute our policy! The use of a lazy model allows us to bypass the need to fetch the shape of the observation space, as the model will automatically determine it. This policy is now ready to be run in the environment:

In [8]:
rollout = env.rollout(max_steps=10, policy=policy)
print(rollout)

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 3]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        observation: Tensor(shape=torch.Size([10, 3]), device=cpu, dtype=torch.float32, 

In [9]:
rollout["action"]

tensor([[0.1583],
        [0.1658],
        [0.1751],
        [0.1822],
        [0.1899],
        [0.1923],
        [0.1923],
        [0.1888],
        [0.1862],
        [0.1824]], grad_fn=<StackBackward0>)

It's not a great policy, as it's based on a model that is randomly initalized... let's see how it does by feeding it some data and seeing what actions it selects. 

In [10]:
print(policy(rollout["observation"]))


tensor([[0.1583],
        [0.1658],
        [0.1751],
        [0.1822],
        [0.1899],
        [0.1923],
        [0.1923],
        [0.1888],
        [0.1862],
        [0.1824]], grad_fn=<AddmmBackward0>)


So the policy just takes the observations and uses a model to set the action to take. In this case it's a force applied to the end of the pendulm. 

We'll see that depending on the type of actions that the agent can take in the environment we'll have to build slightly different models. E.g., if the actions are left, right, nothing, then we'd have to build a categorical model. 

Let's try the steps above again this time with a different environment: one that requires discrete actions to be chosen.

### Descrete action spaces

In [11]:
env = GymEnv("CartPole-v1")

#Actions
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")

#Actions
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

Action space: 2
Observation space: 4


So we can see that the action space is two, but what are the values needed for these actions?

In [12]:
env.action_spec

OneHot(
    shape=torch.Size([2]),
    space=CategoricalBox(n=2),
    device=cpu,
    dtype=torch.int64,
    domain=discrete)

So we need a categorical output of size two set as a onehot encoding.
[0,1] Right
[1,0] Left

To do this we'll leave our model the same. It will output two values. These values will be the logits, one per action. The larger the logit the more likely the action. 

Then we'll use a probabilistic actor to choose actions based on the logits.

In [13]:
from torchrl.modules import ProbabilisticActor
from torchrl.modules.distributions import OneHotCategorical

#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
module = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["logits"]          
)

#go from logits to actions using the ProbabilisticActor
actor = ProbabilisticActor(
    module=module,
    in_keys=["logits"],
    out_keys=["action"],
    distribution_class=OneHotCategorical,   # converts logits values to one hot encoding: [0,1] or [1,0]
    return_log_prob=True, # return the log probabilities of the actions. this will be useful later!
)


In [14]:
#use our new actor when doing a rollout
data_rollout = env.rollout(max_steps=10, policy=actor)

#take a look
data_rollout

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_log_prob: Tensor(shape=torch.Size([10]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        logits: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.boo

In [15]:
data_rollout["action"]

tensor([[0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1]])

Here we can see what actions our agent chose.

In [16]:
data_rollout["logits"]

tensor([[-0.0485,  0.1263],
        [-0.0610,  0.1217],
        [-0.0691,  0.1108],
        [-0.0648,  0.1020],
        [-0.0573,  0.0957],
        [-0.0534,  0.0897],
        [-0.0566,  0.0895],
        [-0.0644,  0.0928],
        [-0.0739,  0.0939],
        [-0.0841,  0.0938]], grad_fn=<StackBackward0>)

We can see that our model is just choosing the action with the highest logits.

We might run into trouble here, as if the agent always chooses the highest logit, then the agent will not explore other options... let's see how we can let these kinds of agents explore.



### Explore vs Exploit

If the agent selects the highest logit it is essentially exploiting it's existing knowledge and choosing the best action it knows. 

If the agent instead selects a random action it is essentially exploring potential actions to see if there might be a better option out there. 

We need to know how to control this explore vs exploit behaviour, especially for environments with categorial action spaces.

Let's learn abou the e-greedy option, it'll let us:

> Set how often the agent choose a random action. We'll call this probability of random action epsilon (eps)

> We can even set it up so that the agent starts off choosing random actions often, then as it learns better actions it can focus more on those better actions.

> This way it can start off exploring then start to exploit more as it learns more from its environment.

In [17]:
from tensordict.nn import TensorDictSequential
from torchrl.modules import EGreedyModule
from torchrl.envs.utils import ExplorationType, set_exploration_type



exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=1000, 
    eps_init=0.5
)

In [18]:
policy_explore = TensorDictSequential(actor, exploration_module)

with set_exploration_type(ExplorationType.RANDOM):
    rollout_explore = env.rollout(max_steps=10, policy=policy_explore)

rollout_explore

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_log_prob: Tensor(shape=torch.Size([10]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        logits: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.boo

In [19]:
rollout_explore["action"]

tensor([[1, 0],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [1, 0],
        [0, 1]])

In [21]:
rollout_explore["logits"]

tensor([[-0.0471,  0.1267],
        [-0.0391,  0.1265],
        [-0.0250,  0.1507],
        [-0.0397,  0.1272],
        [-0.0254,  0.1513],
        [-0.0402,  0.1285],
        [-0.0254,  0.1521],
        [-0.0408,  0.1302],
        [-0.0247,  0.1533],
        [ 0.0081,  0.1766]], grad_fn=<StackBackward0>)

### End

Now we know how to setup environments, build policies, and use these policies to guide the actions of our agents. 

But our policies are really bad at the moment... let's look next at how to allow these policies to get better (i.e., how do we get our agents to learn?!)

**Things to try**


> Check out the official tutorial [here](https://docs.pytorch.org/rl/stable/tutorials/getting-started-1.html)
